# 파일럿 테스트

- 목표: 각 공고에 대해 NCS 세분류 Top-3 후보와 점수를 뽑아 pilot_top3.csv를 만듭니다.

- 입력
    - job_postings_processed.csv : job_posting_id, text
    - ncs_roles_processed.csv : ncs_id(job_role_id), role_text, large/mid/small, job_role_name
- 출력
    - pilot_top3.csv : 공고 1개당 Top-3 후보(계층 포함)
    - 파일럿은 SBERT만 먼저 돌리는 게 가장 안정적입니다.

In [1]:
# CUDA 사용 가능 여부 확인
import torch
print(torch.cuda.is_available())

True


In [3]:
if torch.cuda.is_available():
    print(f"현재 GPU 장치: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"사용 가능한 GPU 개수: {torch.cuda.device_count()}")
else:
    print("GPU를 찾을 수 없습니다. 드라이버나 런타임 설정을 확인하세요.")

현재 GPU 장치: NVIDIA GeForce GTX 1050
CUDA 버전: 12.4
사용 가능한 GPU 개수: 1


- SBERT 모델 다운로드

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")
print("모델 로드 성공")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 로드 성공


- 모델 실행

In [7]:
import os
import numpy as np
import pandas as pd

# ====== 경로 설정 (필요시 수정) ======
JOB_PATH = "datas/processed/job_postings_processed.csv"   # job_posting_id, text 포함
NCS_PATH = "datas/processed/ncs_roles_processed.csv"     # ncs_id(or job_role_id), role_text, 계층컬럼 포함
OUT_PATH = "outputs/sbert_final_top3.csv"

# ====== SBERT 모델 설정 ======
# 한국어 성능/가성비가 무난한 공개 모델 예시입니다.
MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

TOP_K = 3
BATCH_SIZE = 32

# ====== Sentence-Transformers 로드 ======
try:
    from sentence_transformers import SentenceTransformer
except ImportError as e:
    raise ImportError(
        "sentence-transformers가 필요합니다.\n"
        "설치: pip install -U sentence-transformers\n"
    ) from e

# ====== 1) 데이터 로드 ======
job = pd.read_csv(JOB_PATH)
ncs = pd.read_csv(NCS_PATH)

# ---- job 필수 컬럼 체크 ----
if "job_posting_id" not in job.columns:
    raise ValueError("job_postings_processed.csv에 'job_posting_id' 컬럼이 필요합니다.")
if "text" not in job.columns:
    raise ValueError("job_postings_processed.csv에 'text' 컬럼이 필요합니다.")

# ---- ncs 필수 컬럼 체크 ----
# 전처리 코드에서 ncs_id + role_text로 저장했으므로 기본은 ncs_id를 사용합니다.
ID_COL = None
for cand in ["ncs_id", "job_role_id"]:
    if cand in ncs.columns:
        ID_COL = cand
        break
if ID_COL is None:
    raise ValueError("ncs_roles_processed.csv에 'ncs_id' 또는 'job_role_id' 컬럼이 필요합니다.")

# role_text 컬럼은 임베딩 대상이므로 필수입니다.
if "role_text" not in ncs.columns:
    raise ValueError("ncs_roles_processed.csv에 'role_text' 컬럼이 필요합니다.")

# 계층 컬럼은 있으면 포함, 없으면 자동 생략
HIER_COLS = [c for c in ["large_category", "mid_category", "small_category"] if c in ncs.columns]
print(f"계층 컬럼: {HIER_COLS} (없으면 생략)")
NAME_COL = "job_role_name" if "job_role_name" in ncs.columns else None
print(f"역할명 컬럼: {NAME_COL} (없으면 생략)")

# 결측 텍스트 제거(안전장치)
job["text"] = job["text"].fillna("").astype(str)
ncs["role_text"] = ncs["role_text"].fillna("").astype(str)

# ====== 2) 임베딩 생성 ======
model = SentenceTransformer(MODEL_NAME)

# role 임베딩 (NCS 세분류) : 결과 role_emb: (세분류 개수 × 임베딩 차원) 행렬
role_texts = ncs["role_text"].tolist()
role_emb = model.encode(
    role_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # 코사인 유사도 계산을 dot product로 단순화
)

# job posting 임베딩 
post_texts = job["text"].tolist()
post_emb = model.encode(
    post_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# ====== 3) 유사도 계산 & Top-K 추출 ======
# 유사도 행렬 계산
# normalize_embeddings=True 이므로 cosine similarity = dot product
# 모든 공고가 모든 세분류와 얼마나 비슷한지 한 번에 계산합니다.
sim = post_emb @ role_emb.T  # (num_posts, num_roles)

# Top-K 후보 인덱스만 빠르게 뽑기
top_idx = np.argpartition(-sim, TOP_K - 1, axis=1)[:, :TOP_K]  # Top-K 후보 인덱스(정렬 전)

# 각 row마다 Top-K를 유사도 내림차순 정렬(후보를 점수 기준으로 정렬(진짜 1~3등 만들기))
top_scores = np.take_along_axis(sim, top_idx, axis=1)
order = np.argsort(-top_scores, axis=1)
top_idx_sorted = np.take_along_axis(top_idx, order, axis=1)
top_scores_sorted = np.take_along_axis(top_scores, order, axis=1)

# ====== 4) 결과를 long-format (행 = post_id x rank) 으로 저장 ======
rows = []
ncs_id_list = ncs[ID_COL].astype(str).tolist()

for i, post_id in enumerate(job["job_posting_id"].astype(str).tolist()):
    for r in range(TOP_K):
        j = int(top_idx_sorted[i, r])
        score = float(top_scores_sorted[i, r])

        row = {
            "job_posting_id": post_id,
            "rank": r + 1,
            "job_role_id": ncs_id_list[j],     # 세분류 ID
            "score": score,
        }

        if NAME_COL:
            row["job_role_name"] = ncs.iloc[j][NAME_COL]

        # 계층 정보가 있으면 포함
        for hc in HIER_COLS:
            row[hc] = ncs.iloc[j][hc]

        rows.append(row)

pilot_top3 = pd.DataFrame(rows)

# ====== 5) 저장 ======
pilot_top3.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[OK] saved: {OUT_PATH} | rows={len(pilot_top3)} (={len(job)} postings x {TOP_K})")



계층 컬럼: ['large_category', 'mid_category', 'small_category'] (없으면 생략)
역할명 컬럼: job_role_name (없으면 생략)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[OK] saved: outputs/sbert_final_top3.csv | rows=921 (=307 postings x 3)


In [10]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('outputs/sbert_final_top3.csv')

# 2. rank가 1인 데이터만 필터링
rank1_df = df[df['rank'] == 1]

# 3. job_role_id별로 빈도수를 계산하되, job_role_name을 함께 유지하기 위해 
# 두 컬럼 모두를 기준으로 그룹화(groupby)합니다.
# .size()는 그룹별 행의 개수를 세고, .reset_index()는 결과를 다시 데이터프레임 형태로 만듭니다.
result = rank1_df.groupby(['job_role_id', 'job_role_name']).size().reset_index(name='count')

# 4. 빈도수(count) 기준 내림차순 정렬
result = result.sort_values(by='count', ascending=False).reset_index(drop=True)

# 5. 결과 확인
result

# 6. (선택사항) 결과 저장
# result.to_csv('job_role_counts_rank1.csv', index=False)

,job_role_id,job_role_name,count
0,20010706,인공지능학습데이터구축,200
1,20010703,인공지능모델링,32
2,20010701,인공지능플랫폼구축,27
3,20010707,생성형AI엔지니어링,25
4,20010704,인공지능서비스운영관리,10
5,20010705,인공지능서비스구현,7
6,20010702,인공지능서비스기획,6


- 전체 공고가 전부 7개의 직무 세분류 각 40~50 정도 분포
- Top-1 쏠림(특정 세분류로 과도하게 몰림) 확인

In [ ]:
# ====== (선택) 빠른 점검: Top-1 분포(쏠림 확인) ======
import pandas as pd
top1 = pilot_top3[pilot_top3["rank"] == 1]
print("\n[CHECK] Top-1 distribution (top 10):")


FINAL_PATH = "outputs/sbert_final_top3.csv"

df = pd.read_csv(FINAL_PATH)

# 1) Top-1 분포(쏠림 확인)
top1 = df[df["rank"] == 1]
print("[CHECK] Top-1 distribution (count):")
print(top1["job_role_name"].value_counts())

print("\n[CHECK] Top-1 distribution (ratio):")
print((top1["job_role_name"].value_counts(normalize=True) * 100).round(1))

# 2) '생성형AI엔지니어링'이 Top-3에 포함되는 비율(=Top-3 hit rate)
TARGET = "생성형AI엔지니어링"

# 공고별로 Top-3 후보 리스트를 만들고 포함 여부 체크
hits = (
    df.groupby("job_posting_id")["job_role_name"]
      .apply(list)
      .apply(lambda lst: TARGET in lst)
)

print(f"\n[CHECK] Top-3 hit rate for '{TARGET}': {hits.mean()*100:.1f}% ({hits.sum()}/{len(hits)})")

# 3) 포함될 때 rank가 어디에 위치하는지 분포
# (생성형AI엔지니어링이 등장한 행만)
target_rows = df[df["job_role_name"] == TARGET]

if len(target_rows) == 0:
    print(f"\n[WARN] '{TARGET}'가 Top-3 결과에 한 번도 등장하지 않습니다. (role_text 구성/범위 점검 필요)")
else:
    print(f"\n[CHECK] '{TARGET}' rank distribution:")
    print(target_rows["rank"].value_counts().sort_index())

    # 공고별로 '생성형AI엔지니어링'의 최소 rank(있다면) 확인
    min_rank = target_rows.groupby("job_posting_id")["rank"].min()
    print("\n[CHECK] min rank stats (lower is better):")
    print(min_rank.describe())

# 4) (선택) 점수 분포 확인: TARGET 점수 vs Top-1 점수
# 각 공고에서 rank1 점수
top1_score = df[df["rank"] == 1].set_index("job_posting_id")["score"]

# 각 공고에서 TARGET 점수(없으면 NaN)
target_score = df[df["job_role_name"] == TARGET].set_index("job_posting_id")["score"]

score_compare = pd.DataFrame({
    "top1_score": top1_score,
    "target_score": target_score
})

print("\n[CHECK] Score compare head (top1 vs target):")
print(score_compare.head(10))

print("\n[CHECK] How often target_score is missing (not in Top-3):")
print(score_compare["target_score"].isna().mean() * 100, "%")


[CHECK] Top-1 distribution (top 10):
[CHECK] Top-1 distribution (count):
job_role_name
인공지능학습데이터구축    200
인공지능모델링         32
인공지능플랫폼구축       27
생성형AI엔지니어링      25
인공지능서비스운영관리     10
인공지능서비스구현        7
인공지능서비스기획        6
Name: count, dtype: int64

[CHECK] Top-1 distribution (ratio):
job_role_name
인공지능학습데이터구축    65.1
인공지능모델링        10.4
인공지능플랫폼구축       8.8
생성형AI엔지니어링      8.1
인공지능서비스운영관리     3.3
인공지능서비스구현       2.3
인공지능서비스기획       2.0
Name: proportion, dtype: float64

[CHECK] Top-3 hit rate for '생성형AI엔지니어링': 41.5% (113/272)

[CHECK] '생성형AI엔지니어링' rank distribution:
rank
1    25
2    41
3    57
Name: count, dtype: int64

[CHECK] min rank stats (lower is better):
count    113.000000
mean       2.256637
std        0.788067
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        3.000000
Name: rank, dtype: float64


ValueError: cannot reindex on an axis with duplicate labels

해석 기준(간단)
- Top-3 hit rate가 너무 낮다(예: 30% 이하) → job_role_desc/role_text 구성 문제 가능성 큼
- Top-1이 특정 세분류로 90% 이상 쏠림 → 세분류 간 설명 텍스트가 분별력이 부족하거나, 공고 텍스트가 특정 패턴으로 편향됨 가능성
- 생성형AI엔지니어링이 자주 rank2/3에만 있다 → 모호성은 있지만 Top-3 설계 자체는 유효 (정상 범주)
- 원하시면, 위 점검 결과를 바탕으로 role_text(직무정의+능력단위명) 구성을 어떻게 손봐야 점수가 안정적으로 올라가는지도 바로 연결해드릴 수 있습니다.